# Edge Qwen feedback (S9 data + S3 kappa labels)

Step `edge-qwen-feedback`. Thin wrapper; logic lives in `Scheduler.training.edge.qwen_feedback`.


In [ ]:
import os, time, warnings
os.environ.update(PYTHONWARNINGS='ignore', PYDEVD_DISABLE_FILE_VALIDATION='1', TOKENIZERS_PARALLELISM='false',
                  HF_HUB_DISABLE_PROGRESS_BARS='1', TQDM_DISABLE='1')
warnings.filterwarnings('ignore')
T0 = time.time()

In [ ]:
import os, subprocess
REPO, URL, BRANCH = "/kaggle/working/repo", "https://github.com/saltypal/SemanticSchedulerEnd2End.git", "main"
def _git(*a, cwd=None):
    return subprocess.run(["git", *a], cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()
if os.path.isdir(REPO + "/.git"):
    _git("fetch", "-q", "--depth", "1", "origin", BRANCH, cwd=REPO)
    _git("reset", "-q", "--hard", "FETCH_HEAD", cwd=REPO)
else:
    _git("clone", "--depth", "1", "-q", "--no-progress", "--branch", BRANCH, URL, REPO)
GIT_SHA = _git("rev-parse", "HEAD", cwd=REPO)
print("repo @", GIT_SHA[:8], "(" + _git("log", "-1", "--format=%s", cwd=REPO) + ")")

In [ ]:
import subprocess, time
_t = time.time()
LOG = "/kaggle/working/pip_install.log"
with open(LOG, "w") as _f:
    _rc = subprocess.run(["pip", "install", "-q", "--no-input", "--disable-pip-version-check",
                          "--ignore-requires-python", "-e", ".", *['transformers>=4.49.0', 'accelerate', 'bitsandbytes>=0.43.0', 'PyYAML']],
                         cwd=REPO, stdout=_f, stderr=subprocess.STDOUT).returncode
if _rc:
    print("".join(open(LOG).readlines()[-25:]))
    raise RuntimeError("pip install failed (rc=%s)" % _rc)
print("installed in %.0fs" % (time.time() - _t))

In [ ]:
import os, subprocess
ARGV = ['python', '-u', '-m', 'Scheduler.training.edge.qwen_feedback', '--mode', 'pilot']
print("$ python -m Scheduler.training.edge.qwen_feedback " + " ".join(ARGV[4:]))
_p = subprocess.Popen(ARGV, cwd=REPO, env=dict(os.environ), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end="", flush=True)
if _p.wait():
    raise RuntimeError("Scheduler.training.edge.qwen_feedback exited with %s" % _p.returncode)

In [ ]:
import datetime, json, os, platform, subprocess, time
STEP, ARGS = "edge-qwen-feedback", ['--mode', 'pilot']
def _gpu():
    try:
        return subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
    except Exception:
        return None
try:
    import torch; torch_version = torch.__version__
except Exception:
    torch_version = None
out = "/kaggle/working/runs/" + STEP
os.makedirs(out, exist_ok=True)
manifest = {"step": STEP, "entrypoint": "Scheduler.training.edge.qwen_feedback", "args": ARGS, "git_sha": GIT_SHA,
            "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "python": platform.python_version(),
            "torch": torch_version, "gpu": _gpu(), "wall_time_s": time.time() - T0}
json.dump(manifest, open(out + "/run_manifest.json", "w"), indent=2)
print(manifest)